# **Explanatory Analysis using Stepwise Regression (A Case Study on My Health*e*Vet Customer Satisfaction)**

## Background
The VA's patient portal site, My HealtheVet (MHV), offers veterans 24/7 access to anything related to the healthcare they receive from the VA. During their time on the site, a random sample of users are asked to complete a short survey about their experience. We take this feedback and report to our customer any interesting findings to measure the success of MHV while identifying potential risks or opportunities.

The survey is distributed by ForeSee, a third-party vendor. The survey questions can be organized into six site elements: Satisfaction, Look & Feel, Navigation, Site Information, Site Performance, & Task Process. The survey has three questions for each site element where users grade specific parts of their experience with a scale from 1 to 10. ForeSee uses these responses to calculate a composite score for each site element to summarize the data (the calculation itself is proprietary information, so we do not have access to really know how these composite scores are generated). For our customer, the composite satisfaction score is the most important metric regarding survey feedback.

## Problem

The questions that make up the satisfaction score are too vague, as it generally asks users if the site meets their standards or if they are 'satisfied' with their experience. Although these are important questions to ask, it is difficult to determine what actions management needs to take as the language of the questions are too broad. We want to analyze if there is a significant relationship between the composite satisfaction score to the other questions in the survey. Once we find these relationships, we can find the most influential survey questions and determine which site elements should management prioritize.

## Imported Packages

We imported pandas and numpy as they are excellent packages for cleaning and analyzing data. Importing a calendar was also necessary for handling the date fields in our data (we will get into that more later in the paper). Statsmodels is another data analysis package full of various statistical procedures, and for this project we used its capabilities in linear regression.

In [ ]:
# Import packages
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import calendar

/usr/local/lib/python3.7/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


## Importing Data & Approach

The survey feedback can be downloaded as a CSV file from the vendor. We want to use a year's worth of data for our analysis so our timeline for the table below is from August 2020 to July 2021.

Our goal is to reduce the number of variables to identify the most significant predictors. Creating a simpler data model allows us to best explain our response (dependent variable) using the least number of predictors (independent variables).

To reduce the number of variables, we will be using a procedure called Stepwise Regression. Stepwise Regression is a method of finding which predictors are significant in predicting the response. There are multiple methods within Stepwise Regression, so for this project we opted to use Backwards Elimination as it is the generally preferred method. Backwards Elimination is when you fit all of your predictors against the response variable into a model but removing them one-by-one until you're left with only your significant predictors.

Below is the data table we will use for our analysis, with each row being a survey response and each column being a survey question, composite score, or any other information about the survey.

In [ ]:
# Load Data
surveyData = pd.read_excel("/content/survey_data.xlsx")
surveyData.head()

,Satisfaction,Date,Look and Feel,Navigation,Site Information,Site Performance,Task Process,Recommend,Return,Trust - Level MHV,Trust - Level VA,Use Web Channel Over Others,Look and Feel - Appeal,Look and Feel - Balance,Look and Feel - Readability,Navigation - Layout,Navigation - Options,Navigation - Organized,Recommend_1,Return_2,Satisfaction - Expectations,Satisfaction - Ideal,Satisfaction - Overall,Site Information - Answers,Site Information - Thoroughness,Site Information - Understandable,Site Performance - Completeness,Site Performance - Consistency,Site Performance - Loading,Task Process - Efficiency,Task Process - Procedures,Task Process - Time,Trust - Level MHV_3,Trust - Level VA_4,Use Web Channel Over Others_5,ANGender,browser_name,Standardized Age Range
0,7,2021-07-01,66,5,23,72,28,0,89,44,67,11,7.0,NaN,7.0,2.0,NaN,1.0,1.0,9.0,1.0,3.0,1.0,1.0,5.0,NaN,6.0,NaN,9.0,3.0,NaN,4.0,5.0,7.0,2.0,Male,Safari,30-39
1,48,2021-07-01,55,44,28,67,38,78,89,78,67,44,5.0,NaN,7.0,5.0,NaN,5.0,8.0,9.0,5.0,5.0,6.0,2.0,5.0,NaN,7.0,NaN,7.0,7.0,NaN,2.0,8.0,7.0,5.0,Male,Safari,70-79
2,89,2021-07-01,89,89,90,78,83,100,100,89,100,100,9.0,NaN,9.0,9.0,NaN,9.0,10.0,10.0,9.0,9.0,9.0,9.0,9.0,NaN,8.0,NaN,8.0,8.0,NaN,9.0,9.0,NaN,10.0,Male,Chrome,70-79
3,78,2021-07-01,95,89,82,95,88,100,100,89,78,67,9.0,10.0,NaN,NaN,9.0,9.0,10.0,10.0,8.0,7.0,9.0,NaN,8.0,9.0,NaN,9.0,10.0,NaN,9.0,9.0,9.0,8.0,7.0,Male,Edge,70-79
4,96,2021-07-01,94,94,90,94,94,100,100,89,89,100,9.0,NaN,10.0,10.0,NaN,9.0,10.0,10.0,10.0,9.0,10.0,9.0,9.0,NaN,10.0,NaN,9.0,9.0,NaN,10.0,9.0,9.0,10.0,Male,IE,70-79


## Data Manipulation: Date Fields

'Date' is a field in our table that marks when a survey was received, and with a regression model, we can determine if a seasonal trend in satisfaction exists using this field. However, statsmodels cannot use this datatype as a variable so we have to transform it into a workable format.

Since reporting is done on a monthly basis, we extract the month of every date, as done below. In the next step, we will convert each month into a binary variable. For example, we will create a variable for January and it will have a value of '1' if a variable was submitted in that month, '0' if it wasn't. Each month will have its own binary variable, with April being the baseline (if a survey was submitted in April, then all date-related binary variables will equal zero).

In [ ]:
# Convert date fields into month names.
surveyData['Date'] = pd.DatetimeIndex(surveyData['Date']).month
surveyData['Date'] = surveyData['Date'].apply(lambda x: calendar.month_abbr[x])
surveyData.head()

,Satisfaction,Date,Look and Feel,Navigation,Site Information,Site Performance,Task Process,Recommend,Return,Trust - Level MHV,Trust - Level VA,Use Web Channel Over Others,Look and Feel - Appeal,Look and Feel - Balance,Look and Feel - Readability,Navigation - Layout,Navigation - Options,Navigation - Organized,Recommend_1,Return_2,Satisfaction - Expectations,Satisfaction - Ideal,Satisfaction - Overall,Site Information - Answers,Site Information - Thoroughness,Site Information - Understandable,Site Performance - Completeness,Site Performance - Consistency,Site Performance - Loading,Task Process - Efficiency,Task Process - Procedures,Task Process - Time,Trust - Level MHV_3,Trust - Level VA_4,Use Web Channel Over Others_5,ANGender,browser_name,Standardized Age Range
0,7,Jul,66,5,23,72,28,0,89,44,67,11,7.0,NaN,7.0,2.0,NaN,1.0,1.0,9.0,1.0,3.0,1.0,1.0,5.0,NaN,6.0,NaN,9.0,3.0,NaN,4.0,5.0,7.0,2.0,Male,Safari,30-39
1,48,Jul,55,44,28,67,38,78,89,78,67,44,5.0,NaN,7.0,5.0,NaN,5.0,8.0,9.0,5.0,5.0,6.0,2.0,5.0,NaN,7.0,NaN,7.0,7.0,NaN,2.0,8.0,7.0,5.0,Male,Safari,70-79
2,89,Jul,89,89,90,78,83,100,100,89,100,100,9.0,NaN,9.0,9.0,NaN,9.0,10.0,10.0,9.0,9.0,9.0,9.0,9.0,NaN,8.0,NaN,8.0,8.0,NaN,9.0,9.0,NaN,10.0,Male,Chrome,70-79
3,78,Jul,95,89,82,95,88,100,100,89,78,67,9.0,10.0,NaN,NaN,9.0,9.0,10.0,10.0,8.0,7.0,9.0,NaN,8.0,9.0,NaN,9.0,10.0,NaN,9.0,9.0,9.0,8.0,7.0,Male,Edge,70-79
4,96,Jul,94,94,90,94,94,100,100,89,89,100,9.0,NaN,10.0,10.0,NaN,9.0,10.0,10.0,10.0,9.0,10.0,9.0,9.0,NaN,10.0,NaN,9.0,9.0,NaN,10.0,9.0,9.0,10.0,Male,IE,70-79


## Data Manipulation: Identifying Response and Predictor Variables

We drop the composite scores from our table since they are already correlated with the survey questions, as we want to keep our predictors independent from each other. We dropped the questions directly related to satisfaction since those are too broad for understanding causes in satisfaction trends.

We identified the satisfaction score as our response variable, with the remaining columns as our predictors.

In [ ]:
# Define predictors and response variable. Clean up df for regression testing.
x = surveyData.drop(['Satisfaction', 'Use Web Channel Over Others', 'Satisfaction - Expectations', 'Satisfaction - Ideal', 'Satisfaction - Overall', 'Look and Feel',	'Navigation',	'Site Information',	'Site Performance',	'Task Process',	'Recommend',	'Return', 'browser_name', 'Recommend_1'], axis=1)
x = x.fillna(x.mean())
x = sm.add_constant(x)
x = pd.get_dummies(data = x, drop_first=True)
x = x.astype('int')
y = surveyData["Satisfaction"]
x

,const,Trust - Level MHV,Trust - Level VA,Look and Feel - Appeal,Look and Feel - Balance,Look and Feel - Readability,Navigation - Layout,Navigation - Options,Navigation - Organized,Return_2,Site Information - Answers,Site Information - Thoroughness,Site Information - Understandable,Site Performance - Completeness,Site Performance - Consistency,Site Performance - Loading,Task Process - Efficiency,Task Process - Procedures,Task Process - Time,Trust - Level MHV_3,Trust - Level VA_4,Use Web Channel Over Others_5,Date_Aug,Date_Dec,Date_Feb,Date_Jan,Date_Jul,Date_Jun,Date_Mar,Date_May,Date_Nov,Date_Oct,Date_Sep,ANGender_Male,ANGender_Prefer not to respond,Standardized Age Range_30-39,Standardized Age Range_40-49,Standardized Age Range_50-59,Standardized Age Range_60-69,Standardized Age Range_70-79,Standardized Age Range_80 or older,Standardized Age Range_Prefer not to respond,Standardized Age Range_Under 20
0,1,44,67,7,8,7,2,8,1,9,1,5,8,6,8,9,3,7,4,5,7,2,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0
1,1,78,67,5,8,7,5,8,5,9,2,5,8,7,8,7,7,7,2,8,7,5,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,1,89,100,9,8,9,9,8,9,10,9,9,8,8,8,8,8,7,9,9,7,10,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
3,1,89,78,9,10,8,8,9,9,10,7,8,9,8,9,10,7,9,9,9,8,7,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
4,1,89,89,9,8,10,10,8,9,10,9,9,8,10,8,9,9,7,10,9,9,10,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116170,1,100,100,8,10,10,10,10,8,10,10,8,10,10,10,8,7,10,8,6,7,10,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
116171,1,100,100,9,8,8,8,9,8,10,7,10,10,8,10,9,7,8,8,6,7,10,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
116172,1,100,100,5,5,8,8,1,1,8,7,5,4,8,4,5,7,1,1,6,7,10,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
116173,1,100,100,8,10,10,7,9,8,10,8,8,10,10,9,8,9,7,8,6,7,10,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0


## Data Manipulation: Checking for Multicollinearity

Multicollinearity is when two variables are significantly correlated with each other. This can cause a multitude of problems in regression testing as our independent variables won't truly be independent. One way of checking for that is to calculate the Variance Inflation Factor (VIF). A general rule is to keep variables with a VIF of approximiately 5 or below. In our output, our variables do meet this criteria so we can move onto fitting the model.

In [ ]:
# Check Variance Inflation Factor for Multicollinearity between variables
vif_data = pd.DataFrame()
vif_data["Potential Variables"] = x.columns
vif_data["VIF"] = [variance_inflation_factor(x.values, i) for i in range(len(x.columns))]

print(vif_data.sort_values('VIF', ascending=False))

                             Potential Variables         VIF
0                                          const  100.934454
1                              Trust - Level MHV    5.231065
7                           Navigation - Options    5.183745
8                         Navigation - Organized    5.119317
17                     Task Process - Procedures    4.998242
2                               Trust - Level VA    4.822426
12             Site Information - Understandable    4.806015
6                            Navigation - Layout    4.791063
10                    Site Information - Answers    4.316297
18                           Task Process - Time    4.268239
16                     Task Process - Efficiency    3.947614
11               Site Information - Thoroughness    3.935916
4                        Look and Feel - Balance    3.509017
20                            Trust - Level VA_4    3.508411
5                    Look and Feel - Readability    3.435559
3                       

## Data Model: Initial Regression Model

Below is the regression output of all of our variables. From here, we want to reduce the number of variables to remove any insignificant variables.

In [ ]:
# Fit Full Linear Regression Model
model = sm.OLS(y, x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Satisfaction   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.906
Method:                 Least Squares   F-statistic:                 2.662e+04
Date:                Thu, 21 Oct 2021   Prob (F-statistic):               0.00
Time:                        18:06:26   Log-Likelihood:            -4.1578e+05
No. Observations:              116175   AIC:                         8.316e+05
Df Residuals:                  116132   BIC:                         8.321e+05
Df Model:                          42                                         
Covariance Type:            nonrobust                                         
================================================================================================================
                                                   coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
const                                          -61.7985      0.256   -241.761      0.000     -62.300     -61.297
Trust - Level MHV                                0.1315      0.002     56.058      0.000       0.127       0.136
Trust - Level VA                                -0.0194      0.002     -8.634      0.000      -0.024      -0.015
Look and Feel - Appeal                           0.7671      0.028     27.750      0.000       0.713       0.821
Look and Feel - Balance                          0.1709      0.029      5.886      0.000       0.114       0.228
Look and Feel - Readability                      0.1051      0.029      3.663      0.000       0.049       0.161
Navigation - Layout                              1.7576      0.029     61.123      0.000       1.701       1.814
Navigation - Options                             1.3366      0.031     43.188      0.000       1.276       1.397
Navigation - Organized                           1.3978      0.032     44.196      0.000       1.336       1.460
Return_2                                         0.0697      0.021      3.292      0.001       0.028       0.111
Site Information - Answers                       1.8985      0.025     76.083      0.000       1.850       1.947
Site Information - Thoroughness                  1.4686      0.027     55.388      0.000       1.417       1.521
Site Information - Understandable                0.9798      0.029     33.350      0.000       0.922       1.037
Site Performance - Completeness                 -0.0581      0.027     -2.140      0.032      -0.111      -0.005
Site Performance - Consistency                  -0.1727      0.028     -6.138      0.000      -0.228      -0.118
Site Performance - Loading                      -0.0506      0.027     -1.850      0.064      -0.104       0.003
Task Process - Efficiency                        1.2664      0.024     53.581      0.000       1.220       1.313
Task Process - Procedures                        2.5754      0.027     95.475      0.000       2.523       2.628
Task Process - Time                              1.7711      0.027     66.019      0.000       1.719       1.824
Trust - Level MHV_3                             -0.0766      0.022     -3.506      0.000      -0.119      -0.034
Trust - Level VA_4                               0.0374      0.028      1.319      0.187      -0.018       0.093
Use Web Channel Over Others_5                    1.0468      0.020     52.778      0.000       1.008       1.086
Date_Aug                                         0.1603      0.127      1.258      0.208      -0.089       0.410
Date_Dec                                         0.0059      0.122      0.048      0.962      -0.234       0.246
Date_Feb                        

## Data Manipulation: Backwards Elimination

Here we conduct the backwards elimination process. From the coding perspective, it's simply an infinite loop checking the variable with the highest p-value and determining if it's within our 5% level of significance. If a variable fails this criteria, it is removed from our regression model and then we refit the model. The iterative process will stop once the next variable with the highest p-value is within the level of significance.

At the conclusion of this step, it will output the final regression model which we will use to conclude our analysis.

In [ ]:
# Backwards Elimination
alpha = 0.05

while True:
  lm = sm.OLS(y, x).fit()
  if lm.pvalues.sort_values(ascending=False).iloc[0] > alpha:
    insignificant_var = str(lm.pvalues.sort_values(ascending=False).index[0])
    del x[insignificant_var]
  else:
    break
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Satisfaction   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.906
Method:                 Least Squares   F-statistic:                 4.861e+04
Date:                Thu, 21 Oct 2021   Prob (F-statistic):               0.00
Time:                        18:06:35   Log-Likelihood:            -4.1579e+05
No. Observations:              116175   AIC:                         8.316e+05
Df Residuals:                  116151   BIC:                         8.319e+05
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
======================================================================================================
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                -61.9089      0.217   -284.740      0.000     -62.335     -61.483
Trust - Level MHV                      0.1309      0.002     56.911      0.000       0.126       0.135
Trust - Level VA                      -0.0172      0.002    -10.562      0.000      -0.020      -0.014
Look and Feel - Appeal                 0.7536      0.026     28.531      0.000       0.702       0.805
Look and Feel - Balance                0.1758      0.029      6.077      0.000       0.119       0.233
Look and Feel - Readability            0.1073      0.029      3.742      0.000       0.051       0.163
Navigation - Layout                    1.7592      0.029     61.210      0.000       1.703       1.816
Navigation - Options                   1.3380      0.031     43.250      0.000       1.277       1.399
Navigation - Organized                 1.3908      0.031     44.267      0.000       1.329       1.452
Return_2                               0.0672      0.021      3.177      0.001       0.026       0.109
Site Information - Answers             1.9016      0.025     76.393      0.000       1.853       1.950
Site Information - Thoroughness        1.4639      0.026     55.436      0.000       1.412       1.516
Site Information - Understandable      0.9816      0.029     33.448      0.000       0.924       1.039
Site Performance - Completeness       -0.0681      0.027     -2.559      0.010      -0.120      -0.016
Site Performance - Consistency        -0.1867      0.027     -6.927      0.000      -0.240      -0.134
Task Process - Efficiency              1.2675      0.024     53.656      0.000       1.221       1.314
Task Process - Procedures              2.5787      0.027     95.871      0.000       2.526       2.631
Task Process - Time                    1.7587      0.026     67.578      0.000       1.708       1.810
Trust - Level MHV_3                   -0.0587      0.018     -3.327      0.001      -0.093      -0.024
Use Web Channel Over Others_5          1.0461      0.020     52.787      0.000       1.007       1.085
Standardized Age Range_50-59          -0.2469      0.108     -2.281      0.023      -0.459      -0.035
Standardized Age Range_60-69          -0.1756      0.086     -2.033      0.042      -0.345      -0.006
Standardized Age Range_70-79          -0.3583      0.078     -4.610      0.000      -0.511      -0.206
Standardized Age Range_80 or older    -0.4785      0.108     -4.424      0.000      -0.690      -0.266
==============================================================================
Omnibus:                    46777.638   Durbin-Watson:                   2.001
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           492297.453
Skew:                          -1.646   Prob(JB):                

# Model Performance

We must check the performance of our model to ensure that it is robust enough to draw meaningful conclusions. One metric we will use is the adjusted r-squared. This metric measures what percentage of the variation in our response can be explained by the variation in our predictors. Ideally you want somewhere between the upper 80% to upper 90% for a good model.

Our regression model passes this benchmark so we can move onto drawing our conclusion. For context: 90.6% of the variation in satisfaction can be explained by the variation in the responses of the survey questions.

In [ ]:
print('Adjusted R-Squared:', lm.rsquared_adj)

Adjusted R-Squared: 0.9058767433694561


## Final Model: Most Significant Predictors/Survey Questions

For each variable in our model, we were able to calculate a coefficient that explains how much satisfaction will change on average, for a one unit increase while holding all others constant (numbers are found on the regression output). Since these questions have users scaling their experience from 1 to 10, each time a user increases their rating for a certain feature by one, we can estimate how that will impact satisfaction scores. To find the questions with the most impact on satisfaction, we just need to find the ones with the largest coefficients. One other thing to pay attention to is the sign of the coefficient: negative values means that an additional unit increase in a variable will lower satisfaction, while positive values signal an increase in satisfaction.


In [ ]:
results = lm.params.sort_values(ascending=False).abs().to_frame()
results['Coefficient Value'] = lm.params
results.head(5)

,0,Coefficient Value
Task Process - Procedures,2.578729,2.578729
Site Information - Answers,1.901593,1.901593
Navigation - Layout,1.759196,1.759196
Task Process - Time,1.758712,1.758712
Site Information - Thoroughness,1.463919,1.463919


Here are the questions in the MHV survey that have the largest impact on satisfaction scores starting from the most significant (a unit increase in these questions also increase satisfaction scores):

•	Please rate the procedures to accomplish tasks on this site.

•	Please rate how well the site’s information provides answers to your questions.

•	Please rate how well the site layout helps you find what you need.

•	Please rate the time it takes to complete task(s) on this site.

•	Please rate the thoroughness of information provided on this site.

